In [ ]:
from bs4 import BeautifulSoup
import datetime
import pandas as pd
from pandas import ExcelWriter
from selenium import webdriver
import random
from time import sleep
import os

print("IT CONSOB Web Scraping Tool v.1.0")

# Assigning current time, output file name and ExcelWriter object
now = datetime.datetime.now()
filename = 'IT CONSOB SQL Ready {}.xlsx'.format(str(now).replace(":", ".")[:-7])
writer = ExcelWriter(filename)

# Assigning the folders that are going to be used in the process
scriptfolder = os.path.dirname(os.path.abspath(__file__))
tempfolder = os.path.join(scriptfolder, 'tempfolder')
os.chdir(scriptfolder)
outputfolder = r'D:\Regulators\output\ready'

# Creating tempfolder if it doesn't exists, emptying in if it does exist
if os.path.exists(tempfolder):
    for temp_file in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, temp_file))
else:
    os.mkdir(tempfolder)

# Starting Chrome driver, set to download files in tempfolder
chromeOptions = webdriver.ChromeOptions()
prefs = {"plugins.always_open_pdf_externally": True,
         "download.prompt_for_download": False,
         "download.default_directory": tempfolder}
chromeOptions.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()

# Creating dictionary with Regcodes and their respective URLs
regdict = {'IT CONSOB 1': ('https://www.consob.it/web/consob-and-its-activities/class-1-investment-firms-authorised-in-other-eu-countries-with-branches-in-italy', 'Class 1 Investment firms authorised in other EU countries with branches in Italy'),
           'IT CONSOB 2': ('https://www.consob.it/web/consob-and-its-activities/class-1-investment-firms-authorised-in-other-eu-countries-without-branches-in-italy', 'Class 1 investment firms authorised in other EU countries without branches in Italy'),
            'IT CONSOB 3': ('https://www.consob.it/web/consob-and-its-activities/companies-of-non-eu-authorized-to-operate-in-italy-with-branches', 'Companies of non-EU countries other than banks authorized by Consob to operate in Italy with branches'),
            'IT CONSOB 4': ('https://www.consob.it/web/consob-and-its-activities/companies-non-eu-authorized-in-italy-without-branches', 'Companies of non-EU countries other than banks authorized by Consob to operate in Italy without branches'),
            'IT CONSOB 5': ('https://www.consob.it/web/consob-and-its-activities/register-of-italian-investment-firms-sims-', 'Italian investment firms (SIMs)'),
            'IT CONSOB 6': ('https://www.consob.it/web/consob-and-its-activities/investment-firms-with-branches', 'List of Investment Firms authorised in other EU states with branches in Italy'),
            'IT CONSOB 7': ('https://www.consob.it/web/consob-and-its-activities/listed-companies', 'Listed Companies'),
            'IT CONSOB 8': ('https://www.consob.it/web/consob-and-its-activities/mtf-authorised-consob', 'Markets')

           }


# Creating dictionary to containg regulators data and then be converted to a pandas' DataFrame
sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [],
           'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],
           'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [],
           'Address_1': [], 'Address_2': [], 'City': [],
           'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [],
           'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [],
           'RegCtry': [], 'RegCode': [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [],
           'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
           'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [],
           'Zip - Mother company': [], 'Cntry - Mother company': [],
           'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')

for reg, (url, list_name) in regdict.items():
    print('Working with {}'.format(reg))
    driver.get(url)
    sleep(random.uniform(3, 6))

    # Check if CAPTCHA page is displayed and wait for manual solve. 
    # to comment out on production
    while "we apologize for the inconvenience" in driver.page_source.lower():
        print("CAPTCHA detected. Please solve it manually in the browser.")
        input("After solving the CAPTCHA, press Enter to continue...")
        driver.refresh()
        sleep(2)

    # Special handling for list 7
    if reg == "IT CONSOB 7":
        import string
        base_letter_url = "https://www.consob.it/web/consob-and-its-activities/listed-companies/list?startsWith="
        for letter in string.ascii_uppercase:
            letter_url = base_letter_url + letter
            print(f"Processing letter {letter} at {letter_url}")
            driver.get(letter_url)
            sleep(random.uniform(3, 6))
            while "we apologize for the inconvenience" in driver.page_source.lower():
                driver.refresh()
                sleep(2)
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            company_spans = soup.select("span.boxQuotataTitle")
            print(f"Found {len(company_spans)} company names for letter {letter}")
            for span in company_spans:
                name_val = span.get_text(strip=True)
                # Append company info for CONSOB 7
                sqldict['Name'].append(name_val)
                sqldict['LEI Code'].append("")
                sqldict['Address_1'].append("")
                sqldict['City'].append("")
                sqldict['Cntry'].append("")
                sqldict['ListProcessDate'].append(processdate)
                sqldict['ListName'].append(list_name)
                sqldict['RegCtry'].append("IT")
                sqldict['RegCode'].append("CONSOB")
                sqldict['ListCode'].append("7")
                expected_length = len(sqldict['ListProcessDate'])
                for key in sqldict:
                    if len(sqldict[key]) < expected_length:
                        sqldict[key].append("")
    elif reg == "IT CONSOB 8":
        # Special handling for list 8 (Markets)
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        table = soup.find("table", class_="base3")
        if table:
            rows = table.find("tbody").find_all("tr")
            current_company = ""
            for row in rows:
                tds = row.find_all("td")
                if not tds: 
                    continue
                # If the first cell has a <strong> tag, update company name
                strong_tag = tds[0].find("strong")
                if strong_tag:
                    current_company = strong_tag.get_text(strip=True)
                    # Append the company information
                    sqldict['Name'].append(current_company)
                    sqldict['LEI Code'].append("")
                    sqldict['Address_1'].append("")
                    sqldict['City'].append("")
                    sqldict['Cntry'].append("")
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['ListName'].append(list_name)
                    sqldict['RegCtry'].append("IT")
                    sqldict['RegCode'].append("CONSOB")
                    sqldict['ListCode'].append(reg.split(' ')[-1])
                    expected_length = len(sqldict['ListProcessDate'])
                    for key in sqldict:
                        if len(sqldict[key]) < expected_length:
                            sqldict[key].append("")
                else:
                    pass
        else:
            print("No table found for IT CONSOB 8.")
    else:
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        tables = soup.select("div.evidenzalaterale table")
        print(f"Found {len(tables)} tables on the page.")
        for table in tables:
            rows = table.find_all("tr")
            data = {}
            for row in rows:
                tds = row.find_all("td")
                if len(tds) >= 2:
                    label = tds[0].get_text(strip=True).rstrip(":").lower()
                    value = tds[1].get_text(" ", strip=True)
                    if label not in data:
                        data[label] = value
            name_val = data.get("investment firm", "")
            if not name_val:
                name_val = data.get("italian investment firm", "")
            if not name_val:
                print("Available keys in data:", list(data.keys()))
                name_val = data.get("investment firm of non-eu country other than bank", "")
                if not name_val:
                    for key, value in data.items():
                        lower_key = key.lower()
                        if "investment firm" in lower_key and "non-eu" in lower_key:
                            name_val = value
                            print("Found alternate key:", key)
                            break
            lei_code   = data.get("lei code", "")
            reg_office = data.get("registered office", "")
            city       = data.get("city", "")
            country    = data.get("country", "")
            branch     = data.get("branch", "")
            
            sqldict['Name'].append(name_val)
            sqldict['LEI Code'].append(lei_code)
            sqldict['Address_1'].append(reg_office)
            sqldict['City'].append(city)
            sqldict['Cntry'].append(country)
            
            sqldict['ListProcessDate'].append(processdate)
            sqldict['ListName'].append(list_name)
            sqldict['RegCtry'].append('IT')
            sqldict['RegCode'].append('CONSOB')
            sqldict['ListCode'].append(reg.split(' ')[-1])
            
            expected_length = len(sqldict['ListProcessDate'])
            for key in sqldict:
                if len(sqldict[key]) < expected_length:
                    sqldict[key].append("")

os.chdir(scriptfolder)
df = pd.DataFrame(sqldict)
df.to_excel(writer, sheet_name='SQL Ready', index=False)
writer.close()
driver.quit()
    
    
    